# Image vs Edit Mask Explorer (TextMask × ChangeMask)
- CSV columns expected: image, edit_image, prompt
- TextMask: zero-shot CLIPSeg heatmap from (image + short phrases parsed from prompt)
- ChangeMask: per-pixel Lab color delta between image and edit_image
- Intersection: supervision mask M_sup = normalize(TextMask * ChangeMask)
- Visuals + IoU/Dice/area stats, with interactive sliders

Optional: band shaping for hems/cuffs/collar edges.

In [ ]:
# Install if needed
# %pip install -q transformers pillow matplotlib opencv-python pandas ipywidgets
# If you also want DINO later:
# %pip install -q timm
# In Jupyter: enable widgets once per environment
# %pip install -q ipywidgets && jupyter nbextension enable --py widgetsnbextension

In [50]:
# Small vocabulary for fashion parts -> robust phrases for CLIPSeg
PART_PHRASES = {
    "hem":       ["hemline","dress hem","skirt hem"],
    "pant_hem":  ["pant hem","trouser hem"],
    "cuff":      ["cuffs","sleeve cuffs","wrist cuffs"],
    "collar":    ["collar","neckline","lapel"],
    "sleeve":    ["sleeve","long sleeve","short sleeve"],
    "bodice":    ["bodice","torso","upper body"],
    "waist":     ["waist","waistline","belt line"],
    "pocket":    ["pocket","pockets"],
    "button":    ["buttons","button placket"],
    "trim":      ["trim","edge trim","piping"],
    "skirt":     ["skirt"],
    "background":["background","backdrop"]
}

# Extended parts for more sophisticated band detection
PART_PHRASES.update({
    "side_seam":      ["side seams","side seam","side panels","side stripe","side stripes"],
    "side_stripe":    ["side stripe","side stripes","contrast side stripe"],
    "placket":        ["button placket","button strip","front placket"],
    "button_strip":   ["button strip","button tape","button placket"],
    "sleeve_hem":     ["sleeve hem","sleeve edge","sleeve opening"],
    "waistband":      ["waistband","waist band","elastic waist"],
    "pocket_opening": ["pocket opening","pocket edge","pocket mouth"],
    "zipper_tape":    ["zipper tape","zip tape","zipper placket"],
    "lapel_edge":     ["lapel edge","lapel"],
    "hood_edge":      ["hood edge","hood opening"]
})

# Band behavior specifications
BAND_SPECS = {
    "hem":            {"mode": "bottom"},
    "pant_hem":       {"mode": "bottom"},
    "cuff":           {"mode": "end"},
    "collar":         {"mode": "top"},
    "trim":           {"mode": "all_edges"},
    "side_seam":      {"mode": "vertical"},
    "side_stripe":    {"mode": "vertical"},
    "placket":        {"mode": "vertical"},
    "button_strip":   {"mode": "vertical"},
    "sleeve_hem":     {"mode": "end"},
    "waistband":      {"mode": "horizontal"},
    "pocket_opening": {"mode": "end"},
    "zipper_tape":    {"mode": "vertical"},
    "lapel_edge":     {"mode": "end"},
    "hood_edge":      {"mode": "end"},
}

def parts_from_prompt(prompt: str):
    p = prompt.lower()
    hits = []
    for part, variants in PART_PHRASES.items():
        if any(v in p for v in variants):
            hits.append(part)
    # very common garment words that imply parts even if not explicitly named
    if "hem" in p or "hemline" in p:
        if "hem" not in hits: hits.append("hem")
    if "cuff" in p:
        if "cuff" not in hits: hits.append("cuff")
    if "seam" in p:
        if "side_seam" not in hits: hits.append("side_seam")
    if "placket" in p:
        if "placket" not in hits: hits.append("placket")
    if "waistband" in p or "waist band" in p:
        if "waistband" not in hits: hits.append("waistband")
    return list(dict.fromkeys(hits))  # uniq, keep order

In [51]:
# Small vocabulary for fashion parts -> robust phrases for CLIPSeg
PART_PHRASES = {
    "hem":       ["hemline","dress hem","skirt hem"],
    "pant_hem":  ["pant hem","trouser hem"],
    "cuff":      ["cuffs","sleeve cuffs","wrist cuffs"],
    "collar":    ["collar","neckline","lapel"],
    "sleeve":    ["sleeve","long sleeve","short sleeve"],
    "bodice":    ["bodice","torso","upper body"],
    "waist":     ["waist","waistline","belt line"],
    "pocket":    ["pocket","pockets"],
    "button":    ["buttons","button placket"],
    "trim":      ["trim","edge trim","piping"],
    "skirt":     ["skirt"],
    "background":["background","backdrop"]
}

def parts_from_prompt(prompt: str):
    p = prompt.lower()
    hits = []
    for part, variants in PART_PHRASES.items():
        if any(v in p for v in variants):
            hits.append(part)
    # very common garment words that imply parts even if not explicitly named
    if "hem" in p or "hemline" in p:
        if "hem" not in hits: hits.append("hem")
    if "cuff" in p:
        if "cuff" not in hits: hits.append("cuff")
    return list(dict.fromkeys(hits))  # uniq, keep order

In [52]:
# Core helpers: CLIPSeg heatmap, Lab delta, softening, band shaping, overlays, metrics
@torch.no_grad()
def clipseg_heatmap(pil_img, phrases, reduce="mean"):
    if isinstance(phrases, str):
        phrases = [phrases]

    # Get target size from PIL image
    target_size = pil_img.size[::-1]  # (height, width) for OpenCV

    # Process each phrase separately to avoid batch size mismatch
    heatmaps = []
    for phrase in phrases:
        inputs = proc(text=[phrase], images=[pil_img], return_tensors="pt").to(device)
        logits = clipseg(**inputs).logits      # [1, 1, H, W]
        probs = logits.sigmoid()               # [1, 1, H, W]
        heat = probs[0, 0].detach().cpu().numpy()  # [H, W]
        
        # Resize to match original image size
        heat_resized = cv2.resize(heat, (target_size[1], target_size[0]), interpolation=cv2.INTER_LINEAR)
        heatmaps.append(heat_resized)

    # Combine multiple heatmaps
    if len(heatmaps) == 1:
        final_heat = heatmaps[0]
    elif reduce == "mean":
        final_heat = np.mean(heatmaps, axis=0)
    else:  # reduce == "max"
        final_heat = np.maximum.reduce(heatmaps)

    # Normalize
    final_heat = (final_heat - final_heat.min()) / (final_heat.max() - final_heat.min() + 1e-8)
    return final_heat.astype(np.float32)

def lab_delta(img0_rgb: np.ndarray, img1_rgb: np.ndarray, blur=5) -> np.ndarray:
    # Ensure both images have the same size
    if img0_rgb.shape != img1_rgb.shape:
        h, w = min(img0_rgb.shape[0], img1_rgb.shape[0]), min(img0_rgb.shape[1], img1_rgb.shape[1])
        img0_rgb = cv2.resize(img0_rgb, (w, h))
        img1_rgb = cv2.resize(img1_rgb, (w, h))
    
    a = cv2.cvtColor(img0_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    b = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    d = np.linalg.norm(a - b, axis=2)
    if blur > 0:
        d = cv2.GaussianBlur(d, (blur|1, blur|1), 0)
    # robust normalization
    med = np.median(d)
    mad = np.median(np.abs(d - med)) + 1e-6
    z = (d - med) / (1.4826 * mad)
    z = (z - z.min()) / (z.max() - z.min() + 1e-8)
    return z.astype(np.float32)

def soften_mask(heat, keep_thr=0.5, blur_px=5, sharpen=8.0):
    keep = (heat >= keep_thr).astype(np.float32)
    m = heat * keep
    if blur_px > 0:
        m = cv2.GaussianBlur(m, (blur_px|1, blur_px|1), 0)
    med = np.median(m)
    out = 1.0 / (1.0 + np.exp(-sharpen * (m - med)))
    out = (out - out.min()) / (out.max() - out.min() + 1e-8)
    return out.astype(np.float32)

def band_from_region(m_soft: np.ndarray, mode="end", width_px=16):
    """
    Create a narrow band inside a region mask.
    modes:
      - 'bottom','top','left','right' : band near that bbox edge
      - 'vertical'   : bands near left+right bbox edges
      - 'horizontal' : bands near top+bottom bbox edges
      - 'all_edges'  : band around any boundary (thin outline)
      - 'end'        : generic boundary band (good for cuffs/pocket openings)
    """
    m = (m_soft*255).astype(np.uint8)
    if m.max()==0:
        return np.zeros_like(m_soft, dtype=np.float32)

    # bbox
    ys, xs = np.where(m>0)
    y0,y1 = ys.min(), ys.max()
    x0,x1 = xs.min(), xs.max()
    H, W  = m.shape

    band = np.zeros_like(m, dtype=np.uint8)

    if mode in ("all_edges","end"):
        # distance to outside boundary
        dist = cv2.distanceTransform(255-m, cv2.DIST_L2, 3)
        band = ((dist < width_px) & (m > 0)).astype(np.uint8)*255

    elif mode in ("vertical","left","right"):
        xx = np.tile(np.arange(W, dtype=np.int32)[None,:], (H,1))
        if mode in ("vertical","left"):
            band |= (((xx - x0) < width_px) & (m>0)).astype(np.uint8)*255
        if mode in ("vertical","right"):
            band |= (((x1 - xx) < width_px) & (m>0)).astype(np.uint8)*255

    elif mode in ("horizontal","top","bottom"):
        yy = np.tile(np.arange(H, dtype=np.int32)[:,None], (1,W))
        if mode in ("horizontal","top"):
            band |= (((yy - y0) < width_px) & (m>0)).astype(np.uint8)*255
        if mode in ("horizontal","bottom"):
            band |= (((y1 - yy) < width_px) & (m>0)).astype(np.uint8)*255

    # smooth + normalize to [0,1]
    band = cv2.GaussianBlur(band, (5,5), 0)
    out = (band / max(1, band.max())).astype(np.float32)
    return out

# Negative masks to reduce spill
NEG_PHRASES = {
    "skin": ["skin","arm","hand","neck"],
    "face": ["face","head"],
    "hair": ["hair"],
    "logo": ["logo","brand logo","adidas logo","print"]
}

def apply_negatives(pil_img, m_text, scale=0.3):
    neg_list = sum(NEG_PHRASES.values(), [])
    hneg = clipseg_heatmap(pil_img, neg_list, reduce="mean")
    hneg = (hneg - hneg.min())/(hneg.max()-hneg.min()+1e-8)
    m = np.clip(m_text - scale*hneg, 0, 1)
    return m

# Precision/Recall diagnostics
def pr_metrics(text_mask, change_mask, thr=0.5):
    T = (text_mask >= thr).astype(np.uint8)
    C = (change_mask >= thr).astype(np.uint8)
    inter = (T & C).sum()
    prec = inter / (T.sum() + 1e-8)     # spill control
    rec  = inter / (C.sum() + 1e-8)     # coverage of actual change
    return float(prec), float(rec)

def overlay(img_rgb, mask, alpha=0.5, cmap="jet"):
    img = img_rgb.astype(np.float32) / 255.0
    cm  = plt.get_cmap(cmap)
    color = cm(np.clip(mask,0,1))[...,:3]
    return np.clip((1-alpha)*img + alpha*color, 0, 1)

def iou_dice(m1, m2, thr=0.5):
    b1 = (m1 >= thr).astype(np.uint8)
    b2 = (m2 >= thr).astype(np.uint8)
    inter = (b1 & b2).sum()
    union = (b1 | b2).sum()
    iou = inter / (union + 1e-8)
    dice = (2*inter) / (b1.sum() + b2.sum() + 1e-8)
    return float(iou), float(dice), int(b1.sum()), int(b2.sum()), int(inter), int(union)

In [53]:
# (Optional) DINO Δ map — leave commented unless you want it
# import timm, torch.nn.functional as F
# class DinoV2:
#     def __init__(self, device="cuda"):
#         self.m = timm.create_model("dinov2_vitb14", pretrained=True).to(device).eval()
#         self.device=device
#         cfg = timm.data.resolve_data_config({}, model=self.m)
#         self.tfm = timm.data.create_transform(**cfg)
#     @torch.no_grad()
#     def patch_delta(self, pil0, pil1):
#         x0 = self.tfm(pil0).unsqueeze(0).to(self.device)
#         x1 = self.tfm(pil1).unsqueeze(0).to(self.device)
#         with torch.cuda.amp.autocast(dtype=torch.bfloat16 if self.device=="cuda" else torch.float32):
#             f0 = self.m.forward_features(x0)
#             f1 = self.m.forward_features(x1)
#         # try common keys
#         toks0 = f0.get("x_norm_patchtokens", None) or f0.get("tokens", None)
#         toks1 = f1.get("x_norm_patchtokens", None) or f1.get("tokens", None)
#         if toks0 is None or toks1 is None:
#             raise RuntimeError("Unexpected dinov2 forward_features output")
#         if toks0.ndim==3 and toks0.shape[1]==toks1.shape[1]+1:  # drop CLS if present
#             toks0 = toks0[:,1:,:]; toks1 = toks1[:,1:,:]
#         d = (toks1.float() - toks0.float()).norm(dim=-1)  # [1,P]
#         d = (d - d.min())/(d.max()-d.min()+1e-8)
#         g = int(math.sqrt(d.shape[1])+1e-6)
#         dmap = d.view(1,1,g,g)
#         dmap = F.interpolate(dmap, size=pil0.size[::-1], mode="bilinear", align_corners=False)[0,0]
#         out = dmap.detach().cpu().numpy().astype(np.float32)
#         return cv2.GaussianBlur(out, (5,5), 0)
# dino = DinoV2(device=device)

In [54]:
# Load your CSV
CSV_PATH = "/sc/home/felix.boelter/recreategoods/qwen-image-edit-finetune/data/example_image_dataset/train.csv"   # <--- change this
BASE_DIR = "/sc/home/felix.boelter/recreategoods/qwen-image-edit-finetune/data/example_image_dataset/"

df = pd.read_csv(CSV_PATH)
assert set(["image","edit_image","prompt"]).issubset(df.columns), "CSV must have image, edit_image, prompt"

# Fix paths to be absolute
df['image'] = df['image'].apply(lambda x: os.path.join(BASE_DIR, x))
df['edit_image'] = df['edit_image'].apply(lambda x: os.path.join(BASE_DIR, x))

print(f"Loaded {len(df)} samples")
print("Columns:", df.columns.tolist())
print("Sample paths:")
for i in range(min(3, len(df))):
    row = df.iloc[i]
    print(f"  {i}: {os.path.exists(row['image'])}, {os.path.exists(row['edit_image'])}")

Loaded 179813 samples
Columns: ['image', 'edit_image', 'prompt']
Sample paths:
  0: True, True
  1: True, True
  2: True, True


In [55]:
# Interactive viewer with improvements
idx_slider   = widgets.IntSlider(min=0, max=len(df)-1, step=1, value=0, description="row")
thr_text     = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.65, description="Text thr")
thr_change   = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description="Change thr")
blur_px      = widgets.IntSlider(min=0, max=15, step=1, value=5, description="Blur px")
band_parts   = widgets.SelectMultiple(
    options=[
        "hem","pant_hem","cuff","collar","trim",
        "side_seam","side_stripe","placket","button_strip",
        "sleeve_hem","waistband","pocket_opening",
        "zipper_tape","lapel_edge","hood_edge"
    ],
    description="Band parts",
    rows=8
)
band_width   = widgets.IntSlider(min=4, max=32, step=2, value=16, description="Band width")
reduce_mode  = widgets.Dropdown(options=["mean","max"], value="mean", description="Phrase reduce")
show_bands   = widgets.Checkbox(value=False, description="Show band analysis")
use_negatives = widgets.Checkbox(value=True, description="Apply negatives")
neg_scale    = widgets.FloatSlider(min=0.0, max=0.5, step=0.05, value=0.25, description="Neg scale")

ui = widgets.VBox([
    widgets.HBox([idx_slider, thr_text, thr_change]),
    widgets.HBox([blur_px, band_width, reduce_mode]),
    band_parts,
    widgets.HBox([show_bands, use_negatives, neg_scale])
])

def visualize(i, tthr, cthr, blur, band_list, bwidth, reduce_mode, show_band_analysis, apply_negs, neg_scale):
    row = df.iloc[int(i)]
    img = Image.open(row["image"]).convert("RGB")
    edt = Image.open(row["edit_image"]).convert("RGB")
    img_np = np.array(img)
    edt_np = np.array(edt)

    # parse prompt -> parts, gather phrases
    parts = parts_from_prompt(str(row["prompt"]))
    phrases = []
    for p in parts:
        phrases += PART_PHRASES.get(p, [p])
    # fallback: if nothing matched, try a generic garment region to visualize
    if not phrases:
        phrases = ["dress","garment","clothing"]

    # TextMask
    heat = clipseg_heatmap(img, phrases, reduce=reduce_mode)
    M_text_original = soften_mask(heat, keep_thr=tthr, blur_px=blur, sharpen=8.0)

    # Apply smart banding for selected parts
    M_text = M_text_original.copy()
    band_contributions = []
    combined_bands = np.zeros_like(M_text_original)
    
    if parts:
        for p in parts:
            if p in band_list:
                # Get part-specific heatmap
                h_part = clipseg_heatmap(img, PART_PHRASES.get(p,[p]), reduce="mean")
                r_part = soften_mask(h_part, keep_thr=tthr, blur_px=blur, sharpen=8.0)
                # Use smart band mode based on part type
                mode = BAND_SPECS.get(p, {"mode":"end"})["mode"]
                band = band_from_region(r_part, mode=mode, width_px=bwidth)
                band_contributions.append((p, band, mode))
                # Accumulate all bands
                combined_bands = np.maximum(combined_bands, band)
                # Union with existing text mask
                M_text = np.maximum(M_text, band)

    # Apply negative masks to reduce spill
    M_text_before_negs = M_text.copy()
    if apply_negs:
        M_text = apply_negatives(img, M_text, scale=neg_scale)

    # ChangeMask (Lab Δ) with percentile thresholding
    H_lab = lab_delta(img_np, edt_np, blur=blur)
    
    # Use percentile instead of fixed threshold for robustness
    pct = 100 * (1 - cthr)          # if slider is 0.5 -> 50th percentile
    thr_val = np.percentile(H_lab, max(50, pct))  # don't go below median
    keep_change = (H_lab >= thr_val).astype(np.float32)

    # Intersection supervision mask (soft)
    M_sup = (M_text * H_lab)
    M_sup = (M_sup - M_sup.min()) / (M_sup.max() - M_sup.min() + 1e-8)

    # Enhanced stats with precision/recall
    iou_orig, dice_orig, _, _, _, _ = iou_dice(M_text_original, keep_change, thr=min(tthr, 0.5))
    iou_with_bands, dice_with_bands, a_text, a_chg, a_inter, a_union = iou_dice(M_text, keep_change, thr=min(tthr, 0.5))
    
    # Precision/Recall analysis
    prec, rec = pr_metrics(M_text, keep_change, thr=min(tthr, 0.5))
    prec_orig, rec_orig = pr_metrics(M_text_original, keep_change, thr=min(tthr, 0.5))
    
    print(f"row {i} | Original: IoU={iou_orig:.3f} Dice={dice_orig:.3f} Prec={prec_orig:.3f} Rec={rec_orig:.3f}")
    if band_contributions or apply_negs:
        improvements = []
        if band_contributions:
            improvements.append("bands")
        if apply_negs:
            improvements.append("negatives")
        print(f"        | With {'+'.join(improvements)}: IoU={iou_with_bands:.3f} Dice={dice_with_bands:.3f} Prec={prec:.3f} Rec={rec:.3f}")
        print(f"        | Δ: IoU={iou_with_bands-iou_orig:+.3f} Dice={dice_with_bands-dice_orig:+.3f} Prec={prec-prec_orig:+.3f} Rec={rec-rec_orig:+.3f}")
    
    # Quality score for training weight
    quality = 0.5 * prec + 0.5 * rec
    train_weight = 0.2 + 0.8 * np.clip(quality, 0, 1)
    print(f"        | Quality={quality:.3f} → Training weight={train_weight:.3f}")
    print(f"        | Lab Δ percentile threshold: {thr_val:.3f} (pct={pct:.0f}%)")
    
    print("Parts detected:", parts)
    print("Phrases used:", phrases[:5], "..." if len(phrases) > 5 else "")
    if band_contributions:
        band_modes = [f"{p}({mode})" for p, _, mode in band_contributions]
        print("Band modes applied:", band_modes)
    print("prompt:", row["prompt"])

    # Diagnostics
    if prec < 0.15:
        print("💡 LOW PRECISION: TextMask too broad → raise text threshold, enable negatives, use bands")
    if rec < 0.15:
        print("💡 LOW RECALL: TextMask too narrow → lower text threshold, wider bands")
    if quality < 0.2:
        print("⚠️  LOW QUALITY: Consider skipping this sample or using low training weight")

    # Choose layout based on show_band_analysis
    if show_band_analysis and (band_contributions or apply_negs):
        # Extended layout showing improvements
        fig, axes = plt.subplots(3, 3, figsize=(15, 12))
        
        # Row 1: Basic inputs
        axes[0,0].imshow(img_np); axes[0,0].set_title("Input image"); axes[0,0].axis("off")
        axes[0,1].imshow(edt_np); axes[0,1].set_title("Edited image"); axes[0,1].axis("off")
        axes[0,2].imshow(H_lab, cmap="magma"); axes[0,2].set_title(f"ChangeMask (Lab Δ)\nThr: {thr_val:.3f}"); axes[0,2].axis("off")
        
        # Row 2: Text mask evolution
        axes[1,0].imshow(overlay(img_np, M_text_original, alpha=0.6, cmap="viridis")); 
        axes[1,0].set_title(f"Original TextMask\nP={prec_orig:.3f} R={rec_orig:.3f}"); axes[1,0].axis("off")
        
        if band_contributions:
            axes[1,1].imshow(overlay(img_np, combined_bands, alpha=0.7, cmap="plasma")); 
            axes[1,1].set_title(f"Bands ({len(band_contributions)} parts)"); axes[1,1].axis("off")
        elif apply_negs:
            neg_contribution = M_text_before_negs - M_text
            axes[1,1].imshow(overlay(img_np, neg_contribution, alpha=0.7, cmap="Reds")); 
            axes[1,1].set_title(f"Negative mask effect"); axes[1,1].axis("off")
        else:
            axes[1,1].axis("off")
        
        axes[1,2].imshow(overlay(img_np, M_text, alpha=0.6, cmap="viridis")); 
        axes[1,2].set_title(f"Final TextMask\nP={prec:.3f} R={rec:.3f}"); axes[1,2].axis("off")
        
        # Row 3: Results
        axes[2,0].imshow(overlay(img_np, keep_change, alpha=0.6, cmap="hot")); 
        axes[2,0].set_title(f"Change ≥ {thr_val:.3f}"); axes[2,0].axis("off")
        
        axes[2,1].imshow(overlay(img_np, M_sup, alpha=0.6, cmap="cool")); 
        axes[2,1].set_title(f"M_sup\nQuality={quality:.3f}"); axes[2,1].axis("off")
        
        # Show improvement
        improvement = M_text - M_text_original
        axes[2,2].imshow(overlay(img_np, np.abs(improvement), alpha=0.8, cmap="plasma")); 
        axes[2,2].set_title(f"Net Improvement\nΔIoU={iou_with_bands-iou_orig:+.3f}"); axes[2,2].axis("off")
        
    else:
        # Standard compact layout
        fig, axes = plt.subplots(2, 3, figsize=(14, 9))
        axes = axes.ravel()
        
        axes[0].imshow(img_np); axes[0].set_title("Input image"); axes[0].axis("off")
        axes[1].imshow(edt_np); axes[1].set_title("Edited image"); axes[1].axis("off")
        axes[2].imshow(H_lab, cmap="magma"); axes[2].set_title(f"ChangeMask\n(thr: {thr_val:.3f})"); axes[2].axis("off")
        
        improvements = []
        if band_contributions: improvements.append("bands")
        if apply_negs: improvements.append("negs")
        title_suffix = f" (+{'+'.join(improvements)})" if improvements else ""
        axes[3].imshow(overlay(img_np, M_text, alpha=0.5)); 
        axes[3].set_title(f"TextMask{title_suffix}\nP={prec:.3f} R={rec:.3f}"); axes[3].axis("off")
        
        axes[4].imshow(overlay(img_np, keep_change, alpha=0.5)); 
        axes[4].set_title(f"Change mask"); axes[4].axis("off")
        
        axes[5].imshow(overlay(img_np, M_sup, alpha=0.5)); 
        axes[5].set_title(f"M_sup\nQ={quality:.3f} W={train_weight:.3f}"); axes[5].axis("off")
    
    plt.tight_layout(); plt.show()

out = widgets.interactive_output(
    lambda row, Text_thr, Change_thr, Blur_px, Band_parts, Band_width, Reduce, Show_bands, Use_negs, Neg_scale: visualize(
        row, Text_thr, Change_thr, Blur_px, Band_parts, Band_width, Reduce, Show_bands, Use_negs, Neg_scale
    ),
    {
        "row": idx_slider,
        "Text_thr": thr_text,
        "Change_thr": thr_change,
        "Blur_px": blur_px,
        "Band_parts": band_parts,
        "Band_width": band_width,
        "Reduce": reduce_mode,
        "Show_bands": show_bands,
        "Use_negs": use_negatives,
        "Neg_scale": neg_scale
    }
)
display(ui, out)

Output()

In [57]:
# Auto-tuning function for optimal parameters
import random

def tune_params(df, K=80, text_thrs=(0.5,0.6,0.7), change_thrs=(0.4,0.5,0.6), blurs=(3,5), band_ws=(12,16,20)):
    """
    Auto-tune parameters on a subset of data to maximize Dice score
    """
    idxs = random.sample(range(len(df)), min(K, len(df)))
    best = None
    results = []
    
    print(f"Tuning on {len(idxs)} samples...")
    total_configs = len(text_thrs) * len(change_thrs) * len(blurs) * len(band_ws)
    config_count = 0
    
    for tt in text_thrs:
        for ct in change_thrs:
            for bp in blurs:
                for bw in band_ws:
                    config_count += 1
                    dices, ious, precs, recs = [], [], [], []
                    
                    for i in idxs:
                        try:
                            row = df.iloc[i]
                            img = Image.open(row["image"]).convert("RGB")
                            edt = Image.open(row["edit_image"]).convert("RGB")
                            img_np, edt_np = np.array(img), np.array(edt)
                            
                            # Get parts and phrases
                            parts = parts_from_prompt(str(row["prompt"]))
                            phrases = sum([PART_PHRASES.get(p,[p]) for p in parts], [])
                            if not phrases: phrases = ["dress","garment","clothing"]
                            
                            # Build text mask
                            heat = clipseg_heatmap(img, phrases, reduce="mean")
                            M_text = soften_mask(heat, keep_thr=tt, blur_px=bp, sharpen=8.0)
                            
                            # Apply bands for common edge parts
                            for p in parts:
                                if p in ("hem","pant_hem","cuff","collar","side_seam","side_stripe","placket","waistband"):
                                    r_part = soften_mask(clipseg_heatmap(img, PART_PHRASES.get(p,[p]), reduce="mean"), keep_thr=tt, blur_px=bp)
                                    mode = BAND_SPECS.get(p,{"mode":"end"})["mode"]
                                    M_text = np.maximum(M_text, band_from_region(r_part, mode=mode, width_px=bw))
                            
                            # Apply negatives
                            M_text = apply_negatives(img, M_text, scale=0.25)
                            
                            # Change mask with percentile threshold
                            H_lab = lab_delta(img_np, edt_np, blur=bp)
                            thr_val = np.percentile(H_lab, max(50, 100*(1-ct)))
                            C_mask = (H_lab >= thr_val).astype(np.float32)
                            
                            # Metrics
                            iou, dice, *_ = iou_dice(M_text, C_mask, thr=min(tt, 0.5))
                            prec, rec = pr_metrics(M_text, C_mask, thr=min(tt, 0.5))
                            
                            dices.append(dice)
                            ious.append(iou)
                            precs.append(prec)
                            recs.append(rec)
                            
                        except Exception as e:
                            continue
                    
                    if dices:  # Only if we have valid results
                        score = float(np.mean(dices))
                        cand = {
                            "text_thr": tt, "change_thr": ct, "blur": bp, "band_w": bw,
                            "dice": score, "iou": float(np.mean(ious)),
                            "precision": float(np.mean(precs)), "recall": float(np.mean(recs)),
                            "quality": 0.5 * float(np.mean(precs)) + 0.5 * float(np.mean(recs)),
                            "n_samples": len(dices)
                        }
                        results.append(cand)
                        
                        if best is None or score > best["dice"]:
                            best = cand
                    
                    if config_count % 10 == 0:
                        print(f"Progress: {config_count}/{total_configs} configs tested")
    
    print(f"\\nTested {len(results)} valid configurations")
    print(f"Best config: {best}")
    
    # Show top 5 results
    sorted_results = sorted(results, key=lambda x: x["dice"], reverse=True)
    print("\\nTop 5 configurations:")
    for i, res in enumerate(sorted_results[:5]):
        print(f"{i+1}. Dice={res['dice']:.3f} IoU={res['iou']:.3f} P={res['precision']:.3f} R={res['recall']:.3f} "
              f"| text_thr={res['text_thr']} change_thr={res['change_thr']} blur={res['blur']} band_w={res['band_w']}")
    
    return best, results

# Example usage (uncomment to run):
# print("Starting parameter tuning...")
best_config, all_results = tune_params(df, K=50)
print(f"\\nRecommended settings:")
print(f"text_thr: {best_config['text_thr']}")
print(f"change_thr: {best_config['change_thr']}")
print(f"blur: {best_config['blur']}")
print(f"band_width: {best_config['band_w']}")

print("Auto-tuning function defined. Uncomment the example usage to run.")

Tuning on 50 samples...
Progress: 10/54 configs tested
Progress: 20/54 configs tested
Progress: 30/54 configs tested
Progress: 40/54 configs tested
Progress: 50/54 configs tested
\nTested 54 valid configurations
Best config: {'text_thr': 0.5, 'change_thr': 0.5, 'blur': 5, 'band_w': 20, 'dice': 0.43689120243823615, 'iou': 0.2885409620087462, 'precision': 0.5715172698313034, 'recall': 0.37206397548602893, 'quality': 0.4717906226586661, 'n_samples': 50}
\nTop 5 configurations:
1. Dice=0.437 IoU=0.289 P=0.572 R=0.372 | text_thr=0.5 change_thr=0.5 blur=5 band_w=20
2. Dice=0.437 IoU=0.289 P=0.572 R=0.372 | text_thr=0.5 change_thr=0.6 blur=5 band_w=20
3. Dice=0.437 IoU=0.288 P=0.572 R=0.372 | text_thr=0.5 change_thr=0.5 blur=5 band_w=16
4. Dice=0.437 IoU=0.288 P=0.572 R=0.372 | text_thr=0.5 change_thr=0.6 blur=5 band_w=16
5. Dice=0.436 IoU=0.288 P=0.572 R=0.371 | text_thr=0.5 change_thr=0.5 blur=5 band_w=12
\nRecommended settings:
text_thr: 0.5
change_thr: 0.5
blur: 5
band_width: 20
Auto-tuni